In [ ]:
import cv2  # Qué: importa OpenCV para las operaciones de procesamiento de imágenes (filtros, espacios de color, morfología).
import numpy as np  # Qué: importa NumPy para crear estructuras numéricas como el kernel de las operaciones morfológicas. Por qué: OpenCV representa imágenes y kernels como arrays de NumPy, así que es una dependencia habitual junto a cv2.

In [ ]:
image = cv2.imread("../imgs/img1.jpg")  # Qué: carga la imagen base a color (BGR) que se reutiliza en todos los filtros del notebook.

In [ ]:
def show_filters(filters):  # Qué: función auxiliar que recorre un diccionario {nombre: imagen_filtrada} y muestra cada resultado en su propia ventana. Por qué: evita repetir el bloque imshow/waitKey/destroyAllWindows para cada filtro individual.

    for filter_name, filtered_image in filters.items():  # Cómo: .items() devuelve pares (clave, valor) del diccionario, así se puede usar el nombre del filtro como título de la ventana.

        cv2.imshow(filter_name, filtered_image)  # Qué: abre una ventana con el nombre del filtro como título y dibuja la imagen resultante.

        cv2.waitKey(0)  # Qué: pausa hasta que el usuario presione una tecla, para poder ver cada filtro antes de pasar al siguiente.
        cv2.destroyAllWindows()  # Qué: cierra la ventana actual antes de mostrar el próximo filtro del diccionario.

# Filtros de color
Estos filtros de color permiten trabajar con la imagen en distintos espacios de color, cada uno con aplicaciones específicas en procesamiento de imágenes, detección de objetos, y análisis de color

In [ ]:
color_filters = {  # Qué: diccionario que agrupa la misma imagen convertida a distintos espacios de color, para comparar resultados con show_filters. Cómo: cv2.cvtColor reproyecta los valores de cada píxel de un espacio de color a otro usando la fórmula de conversión correspondiente (no reordena ni recorta, solo recalcula los canales).
    "Escala de Grises": cv2.cvtColor(image, cv2.COLOR_BGR2GRAY),  # Por qué: reduce a un solo canal de luminancia; base para muchos algoritmos (bordes, umbralización) que no necesitan color.
    "HSV":      cv2.cvtColor(image, cv2.COLOR_BGR2HSV),  # Por qué: separa el matiz (Hue) del brillo/saturación, útil para segmentar por color de forma robusta a cambios de iluminación.
    "HLS":      cv2.cvtColor(image, cv2.COLOR_BGR2HLS),  # Por qué: similar a HSV pero con la luminosidad como eje central; usado cuando se quiere ajustar brillo sin alterar el color.
    "Lab":      cv2.cvtColor(image, cv2.COLOR_BGR2Lab),  # Por qué: espacio perceptualmente uniforme (L = luminosidad, a/b = componentes de color); útil para comparar colores como los percibe el ojo humano.
    "Luv":      cv2.cvtColor(image, cv2.COLOR_BGR2Luv),  # Por qué: otro espacio perceptualmente uniforme, alternativa a Lab, usado en gráficos y análisis de color.
    "YUV":      cv2.cvtColor(image, cv2.COLOR_BGR2YUV),  # Por qué: separa luminancia (Y) de crominancia (U, V); es el espacio típico en compresión de video y TV analógica.
    "YCrCb":    cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb),  # Por qué: variante de YUV usada en compresión JPEG y en detección de piel por su separación clara de luminancia/color.
    "RGB":      cv2.cvtColor(image, cv2.COLOR_BGR2RGB),  # Por qué: reordena los canales de BGR (interno de OpenCV) a RGB (estándar de la mayoría de librerías de visualización como matplotlib).
    "XYZ":      cv2.cvtColor(image, cv2.COLOR_BGR2XYZ)  # Por qué: espacio de color basado en cómo el ojo humano percibe la luz, usado como referencia para convertir entre otros espacios de color.
}

In [ ]:
print(50*"-","\n Estos filtros de color permiten trabajar con la imagen en distintos espacios de color,"+  # Qué: imprime un separador visual (50 guiones) seguido de una explicación del propósito de los filtros de color. Cómo: 50*"-" repite el carácter para formar la línea separadora; la concatenación con + arma el mensaje en dos partes.
            " cada uno con aplicaciones específicas en procesamiento de imágenes, detección de objetos, y análisis de color\n")
show_filters(color_filters)  # Qué: recorre el diccionario color_filters y muestra cada espacio de color en una ventana, uno por uno.


# Filtros de Desenfoque
Incluye los filtros que suavizan la imagen, como el desenfoque promedio, gaussiano, mediano y bilateral

In [ ]:
blur_filters = {  # Qué: diccionario con distintos algoritmos de suavizado/desenfoque aplicados a la misma imagen, para comparar cómo cada uno maneja bordes y ruido.
    "Desenfoque Promedio":  cv2.blur(image, (5, 5)),  # Cómo: cv2.blur promedia los píxeles dentro de una ventana de 5x5. Por qué: es el método más simple y rápido, pero tiende a difuminar bordes de forma pareja.
    "Desenfoque Gaussiano": cv2.GaussianBlur(image, (5, 5), 0),  # Cómo: pondera los píxeles del kernel 5x5 con una distribución gaussiana en vez de un promedio uniforme; sigma=0 hace que OpenCV lo calcule automáticamente a partir del tamaño del kernel. Por qué: produce un desenfoque más natural, dando más peso al píxel central.
    "Desenfoque Mediano":   cv2.medianBlur(image, 5),  # Cómo: reemplaza cada píxel por la mediana de sus vecinos en una ventana de 5x5. Por qué: es muy efectivo para eliminar ruido "sal y pimienta" sin difuminar tanto los bordes como el promedio o el gaussiano.
    "Desenfoque Bilateral": cv2.bilateralFilter(image, 9, 75, 75)  # Cómo: suaviza considerando tanto la cercanía espacial (diámetro 9) como la similitud de color/intensidad (sigmaColor=75, sigmaSpace=75). Por qué: es el único de los cuatro que preserva bordes nítidos mientras suaviza áreas planas, aunque es más costoso computacionalmente.
}

In [ ]:
show_filters(blur_filters)  # Qué: muestra cada uno de los cuatro desenfoques en ventanas sucesivas para comparar sus resultados.


# Filtros de Detección de Bordes
Aquí tenemos filtros que realzan o detectan bordes, como Sobel, Scharr, Canny y Laplaciano

In [ ]:
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Qué: convierte la imagen a escala de grises. Por qué: los detectores de bordes (Sobel, Canny, Laplaciano) trabajan sobre gradientes de intensidad, no sobre color, así que necesitan una entrada de un solo canal.

In [ ]:
edge_detection_filters = {  # Qué: diccionario con distintos algoritmos de detección/realce de bordes aplicados sobre la imagen en gris.
    "Bordes Sobel X":   cv2.Sobel(gray_image, cv2.CV_64F, 1, 0, ksize=5),  # Cómo: calcula la derivada de intensidad en el eje X (dx=1, dy=0) con un kernel de 5x5; cv2.CV_64F conserva valores negativos (gradientes descendentes) que un tipo entero sin signo perdería. Por qué: resalta bordes verticales, donde el cambio de intensidad ocurre horizontalmente.
    "Bordes Sobel Y":   cv2.Sobel(gray_image, cv2.CV_64F, 0, 1, ksize=5),  # Cómo: igual que el anterior pero con dx=0, dy=1, derivando en el eje Y. Por qué: resalta bordes horizontales.
    "Bordes Canny":     cv2.Canny(gray_image, 100, 200),  # Cómo: algoritmo multi-etapa (gradiente + supresión de no-máximos + umbral con histéresis) que usa 100 y 200 como umbrales bajo/alto para decidir qué bordes conservar. Por qué: da como resultado bordes finos y bien definidos, a diferencia de Sobel que da un mapa de gradiente crudo.
    "Realce Laplaciano": cv2.Laplacian(gray_image, cv2.CV_64F)  # Cómo: aplica la segunda derivada (laplaciano) de la imagen, sensible a cambios de intensidad en todas las direcciones a la vez. Por qué: útil para detectar bordes sin tener que combinar Sobel X y Y por separado, aunque es más sensible al ruido.
}

In [ ]:
show_filters(edge_detection_filters)  # Qué: muestra los cuatro resultados de detección de bordes en ventanas sucesivas.

# Filtros Morfológicos
Son operaciones que modifican la estructura de las imágenes, como la erosión, dilatación, apertura, cierre y gradiente morfológico

In [ ]:
kernel = np.ones((5, 5), np.uint8)  # Qué: crea el elemento estructurante (kernel) que usan las operaciones morfológicas. Cómo: una matriz de 5x5 llena de unos, en tipo uint8 (0-255) porque es el tipo que espera cv2.erode/dilate/morphologyEx. Por qué: el tamaño del kernel determina cuánto "crecen" o "encogen" las regiones de la imagen en cada operación.

In [ ]:
morphological_filters = {  # Qué: diccionario con las operaciones morfológicas clásicas, todas basadas en desplazar el kernel sobre la imagen.
    "Erosión":      cv2.erode(image, kernel, iterations=1),  # Cómo: en cada posición, el píxel de salida solo queda "activo" si todo el kernel encaja en la región. Por qué: reduce/erosiona las regiones claras y elimina ruido pequeño.
    "Dilatación":   cv2.dilate(image, kernel, iterations=1),  # Cómo: el píxel de salida se activa si al menos un píxel del kernel coincide con la región. Por qué: expande las regiones claras, útil para rellenar huecos pequeños.
    "Apertura":     cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel),  # Cómo: aplica erosión seguida de dilatación (MORPH_OPEN). Por qué: elimina ruido pequeño (puntos aislados) sin reducir demasiado el tamaño general de los objetos.
    "Cierre":       cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel),  # Cómo: aplica dilatación seguida de erosión (MORPH_CLOSE). Por qué: cierra huecos pequeños dentro de los objetos sin agrandarlos de forma neta.
    "Gradiente Morfológico": cv2.morphologyEx(image, cv2.MORPH_GRADIENT, kernel)  # Cómo: resta la imagen erosionada a la dilatada (dilatación - erosión). Por qué: resalta el contorno/borde de los objetos, similar a un detector de bordes pero basado en morfología.
}

In [ ]:

show_filters(morphological_filters)  # Qué: muestra los cinco resultados morfológicos en ventanas sucesivas.


# Filtro de Ecualización
Mejora el contraste de la imagen mediante la ecualización del histograma

In [ ]:
equalization_filter = {
    "Ecualización de Histograma": cv2.equalizeHist(gray_image)  # Qué: redistribuye los niveles de intensidad de la imagen en gris para que el histograma quede más plano. Cómo: cv2.equalizeHist calcula el histograma acumulado y remapea cada valor de píxel según esa distribución. Por qué: mejora el contraste en imágenes muy oscuras o muy claras, resaltando detalles que antes quedaban comprimidos en pocos niveles de gris. Solo funciona en imágenes de un canal, por eso se usa gray_image y no image.
}


In [ ]:
show_filters(equalization_filter)  # Qué: muestra el resultado de la ecualización de histograma en una ventana.

## 🧪 Práctica
Reforzá lo aprendido en este módulo resolviendo los ejercicios guiados en [`practicas/2_practica.ipynb`](../practicas/2_practica.ipynb).